##1. Environment Setup
This cell sets up the environment by mounting the Google Drive by defining the paths to the UCSD Anomlay Dataset zip folder. Then extracting the dataset if not already present. Printing path of Peds reveal both are accessible.

In [18]:
from google.colab import drive
from pathlib import Path
from zipfile import ZipFile

drive.mount('/content/drive')

DATA_ROOT = Path("/content/UCSD_Anomaly_Dataset/UCSD_Anomaly_Dataset")
PED1_PATH = DATA_ROOT / "UCSDped1"
PED2_PATH = DATA_ROOT / "UCSDped2"

ZIP_PATH = Path("/content/drive/MyDrive/SurveillanceAnomalyDetection/UCSD_Anomaly_Dataset.zip")
EXTRACT_PATH = Path("/content/UCSD_Anomaly_Dataset")

if not (PED1_PATH.exists() and PED2_PATH.exists()):
  with ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_PATH)

print("Ped1:", PED1_PATH.exists(), "| Ped2:", PED2_PATH.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ped1: True | Ped2: True


In [19]:
from google.colab import userdata
from pathlib import Path

GH_TOKEN = userdata.get("GH_PAT")

REPO_DIR = Path(
    "/content/drive/MyDrive/SurveillanceAnomalyDetection/repo"
)

REPO_URL = (
    f"https://{GH_TOKEN}@github.com/"
    "Rishabh-G-Shetye/SurveillanceAnomalyDetection.git"
)

if not REPO_DIR.exists():
    !git clone {REPO_URL} "{REPO_DIR}"
    print("Repository cloned.")
else:
    print("Repository already exists — skipping clone.")

%cd "{REPO_DIR}"

Repository already exists — skipping clone.
/content/drive/MyDrive/SurveillanceAnomalyDetection/repo


##2.Sequence Inventory
Apart from importing the necessary libraries, it defines the helper functions to count the image frames and build an inventory of the dataset. It is made to iterate through the Train and Test splits of both the Ped1 and PEd2 datasets. It then records the information like the sequence name, number of frames and if ground truth masks exist.

In [20]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

IGNORE_NAMES = {".DS_Store", "._.DS_Store"}
FRAME_SUFFIXES = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

def count_frames(seq_dir: Path) -> int:
  # Counting image frames in sequence folder
  return sum(
      1 for p in seq_dir.iterdir()
      if p.is_file() and p.suffix.lower() in FRAME_SUFFIXES
  )
def build_inventory(ped_path: Path, ped_name: str) -> list[dict]:
  #Walk Train/Test  fodler of on PEd dataset
  rows = []
  for split in ("Train", "Test"):
    split_path = ped_path / split
    for seq_dir in sorted(split_path.iterdir()):
      if not seq_dir.is_dir() or seq_dir.name in IGNORE_NAMES:
        continue
      if seq_dir.name.endswith("_gt"):
        continue
      gt_dir = split_path / f"{seq_dir.name}_gt"
      rows.append({
          "dataset": ped_name,
          "split": split,
          "sequence": seq_dir.name,
          "num_frames": count_frames(seq_dir),
          "has_gt": gt_dir.exists(),
          "num_gt_frames": count_frames(gt_dir) if gt_dir.exists() else 0,
      })
  return rows

inventory = build_inventory(PED1_PATH, "Ped1") + build_inventory(PED2_PATH, "Ped2")
df = pd.DataFrame(inventory)
df.head(10)

,dataset,split,sequence,num_frames,has_gt,num_gt_frames
0,Ped1,Train,Train001,200,False,0
1,Ped1,Train,Train002,200,False,0
2,Ped1,Train,Train003,200,False,0
3,Ped1,Train,Train004,200,False,0
4,Ped1,Train,Train005,200,False,0
5,Ped1,Train,Train006,200,False,0
6,Ped1,Train,Train007,200,False,0
7,Ped1,Train,Train008,200,False,0
8,Ped1,Train,Train009,200,False,0
9,Ped1,Train,Train010,200,False,0


##3.Sumary Statistics
This groups the inventory DataFrme by dataset and split to calculate the sumamry statistics such as number of sequences, total frames, average frames per sequence and the number of of sequences with ground truth masks.
The output provides the concise overview of the dataset's composition of both Ped1 and Ped2 distinguishing between the training and testing.

In [21]:
summary = df.groupby(["dataset", "split"]).agg(
    num_sequence=("sequence", "count"),
    total_frames=("num_frames", "sum"),
    avg_frames_per_seq=("num_frames", "mean"),
    num_with_gt=("has_gt", "sum")
).round(1)
summary

num_sequence  total_frames  avg_frames_per_seq  num_with_gt
dataset split                                                             
Ped1    Test             36          7200               200.0           10
        Train            34          6800               200.0            0
Ped2    Test             12          2010               167.5           12
        Train            16          2550               159.4            0

##4. Ground Truth Format inspection
Compares count for number of files and the corresponding frame files, lists few of them, and then prints the size and colour mode of a gt file to confirm format.


In [22]:
sample_gt_dir = PED1_PATH / "Test" / "Test003_gt"
sample_frame_dir = PED1_PATH / "Test" / "Test003"

gt_files = sorted(p for p in sample_gt_dir.iterdir() if p.is_file() and p.suffix.lower() == ".bmp")
frame_files = sorted(p for p in sample_frame_dir.iterdir() if p.is_file() and p.suffix.lower() in FRAME_SUFFIXES)

print("Number of GT files:", len(gt_files), "| Number of frame files:", len(frame_files))
print("Firt few GT files names: ", [p.name for p in gt_files[:5]])

with Image.open(gt_files[0]) as m:
  print("GT sample size/mode:", m.size, m.mode)

Number of GT files: 200 | Number of frame files: 200
Firt few GT files names:  ['001.bmp', '002.bmp', '003.bmp', '004.bmp', '005.bmp']
GT sample size/mode: (238, 158) L


##5. Fixing GT coutning and rebuilding inventory
Rebuilding the inventory building process by introducing GT_SUFFIXES and count_files function to accurate count the groung-truth files. I then rebuilds the df DataFrame with the corrections. Finallt a sanity check is performed to ensure that sequences with ground truth have equal number of frames and GT frames.

In [23]:
GT_SUFFIXES = {".bmp"}

def count_files(dir_path: Path, suffixes: set) -> int:
    """Count files in dir_path matching given suffixes, ignoring OS/system files."""
    if not dir_path.exists():
        return 0
    return sum(1 for p in dir_path.iterdir() if p.is_file() and p.suffix.lower() in suffixes)

def build_inventory(ped_path: Path, ped_name: str) -> list[dict]:
    rows = []
    for split in ("Train", "Test"):
        split_path = ped_path / split
        for seq_dir in sorted(split_path.iterdir()):
            if not seq_dir.is_dir() or seq_dir.name in IGNORE_NAMES or seq_dir.name.endswith("_gt"):
                continue
            gt_dir = split_path / f"{seq_dir.name}_gt"
            rows.append({
                "dataset": ped_name,
                "split": split,
                "sequence": seq_dir.name,
                "num_frames": count_files(seq_dir, FRAME_SUFFIXES),
                "has_gt": gt_dir.exists(),
                "num_gt_frames": count_files(gt_dir, GT_SUFFIXES),
            })
    return rows

inventory = build_inventory(PED1_PATH, "Ped1") + build_inventory(PED2_PATH, "Ped2")
df = pd.DataFrame(inventory)

# sanity check: every sequence with GT should have num_gt_frames == num_frames
mismatches = df[df["has_gt"] & (df["num_gt_frames"] != df["num_frames"])]
print("GT/frame count mismatches:", len(mismatches))
mismatches

GT/frame count mismatches: 0


,dataset,split,sequence,num_frames,has_gt,num_gt_frames


##6. Mask Pixel Value Check


In [24]:
import numpy as np
mask_arr = np.array(Image.open(gt_files[0]))
print("Unique pixel values: ", np.unique(mask_arr))
print("Shape:", mask_arr.shape, "| dtype:", mask_arr.dtype)

Unique pixel values:  [0]
Shape: (158, 238) | dtype: uint8


## 7. Mask Value Scan for Full Seuqnece
Extending the mask pixel value check to the entire sequence of ground-truth files. It iteates through all the GT files in the sample directory and collects all unique pizel values and then identifies the first frame that contains any non-zero pixels.
The output confirms the masks contains both 0 for normal and 255 for anomaly values..

In [25]:
all_values = set()
first_nonzero_frame=None
for i, gt_file in enumerate(gt_files):
  arr = np.array(Image.open(gt_file))
  vals = set(np.unique(arr).tolist())
  all_values |= vals
  if first_nonzero_frame is None and vals != {0}:
    first_nonzero_frame = gt_file.name

print("Unique values across entire sequence", sorted(all_values))
print("First frame with non-zero mask:", first_nonzero_frame)

Unique values across entire sequence [0, 255]
First frame with non-zero mask: 091.bmp


## 8. Image Dimension and Mode Consistency Check

In [26]:
def first_frame_info(seq_dir: Path, suffixes: set) -> tuple:
    """Open the first matching image in seq_dir and return (size, mode)."""
    for p in sorted(seq_dir.iterdir()):
        if p.is_file() and p.suffix.lower() in suffixes:
            with Image.open(p) as img:
                return img.size, img.mode
    return None, None

dim_rows = []
for ped_path, ped_name in [(PED1_PATH, "Ped1"), (PED2_PATH, "Ped2")]:
    for split in ("Train", "Test"):
        split_path = ped_path / split
        for seq_dir in sorted(split_path.iterdir()):
            if not seq_dir.is_dir() or seq_dir.name in IGNORE_NAMES or seq_dir.name.endswith("_gt"):
                continue
            size, mode = first_frame_info(seq_dir, FRAME_SUFFIXES)
            dim_rows.append({"dataset": ped_name, "split": split, "sequence": seq_dir.name,
                              "size": size, "mode": mode})

dim_df = pd.DataFrame(dim_rows)
print("Unique (size, mode) combos per dataset:")
print(dim_df.groupby("dataset")[["size", "mode"]].agg(lambda x: sorted(set(x), key=str)))

Unique (size, mode) combos per dataset:
                 size mode
dataset                   
Ped1     [(238, 158)]  [L]
Ped2     [(360, 240)]  [L]


In [ ]:
%cd "{REPO_DIR}"
!git add notebooks/02_ucsd_dataset_exploration.ipynb
!git commit -m "Rebuild Data"
!git push